# Chapter 8 — Kalman Filter: Bayesian Estimation Over Time

Integrated companion notebook. Preserves the original Chapter 8 Kalman examples from `kalman-filters.ipynb`, adds a transparent single-update calculation, and develops reproducible two-dimensional tracking under a 12-second GPS outage. The original shared notebook remains available for Chapters 9 and 10. All data are synthetic.

**Learning objectives:** Calculate Kalman gain and covariance reduction; distinguish predictions from measurement updates; observe uncertainty growth without GPS; distinguish actual error from model-reported uncertainty.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## Part I — Original Chapter 8 example: sequential 1D motion estimation

The following computation is retained from the original companion notebook.

In [2]:
x, P = 0, 1
Q, R = 0.2, 0.5
z = [1.2, 2.0]
for zi in z:
    x_pred = x + 1
    P_pred = P + Q
    K = P_pred / (P_pred + R)
    x = x_pred + K * (zi - x_pred)
    P = (1 - K) * P_pred
    print(f"Estimate: {x:.3f}, Uncertainty: {P:.3f}")

Estimate: 1.141, Uncertainty: 0.353
Estimate: 2.067, Uncertainty: 0.263


## Part II — Original Chapter 8 example: scalar Kalman gain and covariance

The following computation is also retained from the original companion notebook.

In [8]:
x, P = 0, 1
Q, R = 0.2, 0.5
z = [1.2, 2.0]

for zi in z:
    x_pred = x + 1
    P_pred = P + Q
    K = P_pred / (P_pred + R)
    x = x_pred + K * (zi - x_pred)
    P = (1 - K) * P_pred
    print(f"x={x:.2f}, P={P:.2f}, K={K:.2f}")

x=1.14, P=0.35, K=0.71
x=2.07, P=0.26, K=0.53


## Part III — Single measurement update and Gaussian densities

The following example complements the chapter's analytical single-step update with a graphical comparison of the prediction, measurement likelihood, and posterior. It does not replace the original two-step example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x0, P0, Q, R, z = 0.0, 1.0, 0.1, 0.2, 1.2
x_prior, P_prior = x0, P0 + Q
gain = P_prior / (P_prior + R)
x_post = x_prior + gain * (z - x_prior)
P_post = (1 - gain) * P_prior
print(f"Gain {gain:.6f}, posterior mean {x_post:.6f}, variance {P_post:.6f}")
assert np.isclose(x_post, 1.015384615384615)
assert np.isclose(P_post, 0.169230769230769)

def gaussian_pdf(t, mean, variance):
    return np.exp(-0.5 * (t-mean)**2/variance) / np.sqrt(2*np.pi*variance)

grid = np.linspace(-2, 3, 600)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(grid, gaussian_pdf(grid, x_prior, P_prior), label="Predicted state")
ax.plot(grid, gaussian_pdf(grid, z, R), ls="--", label="Measurement likelihood")
ax.plot(grid, gaussian_pdf(grid, x_post, P_post), label="Updated state", lw=2.4)
ax.set(xlabel="Position (m)", ylabel="Density")
ax.legend(frameon=False); ax.grid(alpha=.2); plt.show()

## Part IV — Two-dimensional tracking and missing GPS

The state is $(x,y,v_x,v_y)$ with a linear constant-velocity process model. Position-only GPS observations are available at 1 Hz except during the **28–40 s outage**; prediction is performed at 0.1 s intervals. The simulated true trajectory includes unmodeled maneuvers, so filter-reported nominal intervals need not achieve their stated coverage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch
from pathlib import Path

rng = np.random.default_rng(20261119)
dt = 0.1
times = np.arange(0, 60 + dt/2, dt)
N = len(times)
F = np.array([[1,0,dt,0], [0,1,0,dt], [0,0,1,0], [0,0,0,1]], float)
G = np.array([[dt*dt/2,0], [0,dt*dt/2], [dt,0], [0,dt]])
qacc = .22
Q = (qacc**2) * G @ G.T + 1e-9 * np.eye(4)
H = np.array([[1,0,0,0], [0,1,0,0]], float)
gps_sd = 2.
R = gps_sd**2 * np.eye(2)
truth = np.zeros((N,4))
truth[0] = [0.,0.,1.15,.45]
for k in range(1,N):
    t = times[k]
    a = np.array([0.,0.])
    if 14 <= t < 22: a = np.array([0., .25])
    elif 22 <= t < 29: a = np.array([-.18, 0.])
    elif 37 <= t < 45: a = np.array([.15, -.22])
    truth[k] = F @ truth[k-1] + G @ (a + rng.normal(0,qacc,2))

fix_idx = np.arange(0,N,10)
valid = np.array([k for k in fix_idx if not (28 <= times[k] < 40)])
gps = np.full((N,2),np.nan)
gps[valid] = truth[valid,:2] + rng.normal(0,gps_sd,(len(valid),2))
x = np.array([1.2,-1.0,.6,.1])
P = np.diag([9.,9.,1.5,1.5])
est = np.zeros_like(truth)
cov = np.zeros((N,4,4))
for k,t in enumerate(times):
    if k:
        x = F@x
        P = F@P@F.T + Q
    if not np.isnan(gps[k,0]):
        innovation = gps[k] - H@x
        S = H@P@H.T+R
        K = np.linalg.solve(S, (P@H.T).T).T
        x = x + K@innovation
        IKH = np.eye(4)-K@H
        P = IKH@P@IKH.T + K@R@K.T
    est[k],cov[k] = x,P
x_err = est[:,0]-truth[:,0]
bound = 1.96 * np.sqrt(cov[:,0,0])
rmse = np.sqrt(np.mean(np.sum((est[:,:2]-truth[:,:2])**2,axis=1)))
print(f"Overall position RMSE: {rmse:.3f} m")

## Figure 8.2 — Tracking and uncertainty

Two equal-width panels use prominent centered headings above the plots, following the Chapter 7 Jupyter figure style. Panel (a) uses a four-line vertical legend; panel (b) wraps the first uncertainty-band label onto two lines. The pale-blue bands and amber hatched GPS outage are visually distinct. The plotted data and intervals are computed above, not manually drawn.

In [ ]:
with plt.rc_context({"font.family":"DejaVu Serif", "font.size":15,
                     "axes.labelsize":16,"xtick.labelsize":13,
                     "ytick.labelsize":13,"pdf.fonttype":42}):
    fig = plt.figure(figsize=(11.5,12.4), facecolor="white")
    ax = fig.add_axes([.115,.535,.77,.395])
    ax2 = fig.add_axes([.115,.120,.77,.270])
    ax.plot(truth[:,0],truth[:,1],lw=2.6,color="#25384A",
            label="True trajectory",zorder=3)
    ax.plot(est[:,0],est[:,1],lw=2.2,color="#B87000",
            label="Kalman estimate",zorder=4)
    ax.scatter(gps[valid,0],gps[valid,1],s=25,facecolors="none",
               edgecolors="#1768A5",lw=1.1,alpha=.75,
               label="GPS observations",zorder=5)
    scale = np.sqrt(-2*np.log(.05)) # sqrt(chi-square_2 95% quantile)
    for k in np.arange(0,N,60):
        vals,vecs = np.linalg.eigh(cov[k,:2,:2])
        vals = np.maximum(vals,0)
        angle = np.degrees(np.arctan2(vecs[1,1],vecs[0,1]))
        ax.add_patch(Ellipse(est[k,:2],2*scale*np.sqrt(vals[1]),
                      2*scale*np.sqrt(vals[0]),angle=angle,
                      facecolor="#B7D5E8",edgecolor="#4D8CAD",
                      lw=.9,alpha=.24,zorder=1))
    for moment, textlabel, offset in [(28,"GPS dropout begins",(9,18)),
                                       (40,"GPS returns",(9,-30))]:
        k = int(round(moment/dt))
        ax.annotate(textlabel,xy=est[k,:2],xytext=offset,
                    textcoords="offset points",fontsize=12,
                    arrowprops={"arrowstyle":"->","lw":.9},zorder=6)
    ax.set(xlabel="East position (m)",ylabel="North position (m)")
    ax.set_xlim(min(truth[:,0].min(),est[:,0].min())-8,
                max(truth[:,0].max(),est[:,0].max())+7)
    ax.set_ylim(min(truth[:,1].min(),est[:,1].min())-7,
                max(truth[:,1].max(),est[:,1].max())+7)
    ax.set_aspect("equal",adjustable="box")
    ax.grid(alpha=.17)
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor="#B7D5E8",edgecolor="#4D8CAD",alpha=.6))
    labels.append("95% position ellipse")
    ax.legend(handles,labels,loc="upper left",ncol=1,
              fontsize=13.2,framealpha=.92)
    ax.set_title("(a) Two-dimensional tracking with GPS dropout",
                 fontsize=19,fontweight="bold",pad=16)

    ax2.fill_between(times,-bound,bound,facecolor="#7FB1D2",alpha=.31,
           label="Nominal 95% interval\n(east-position error)",zorder=2)
    ax2.plot(times,x_err,color="#154E77",lw=2.3,
           label="Estimated minus true east position",zorder=4)
    ax2.axhline(0,color="#364957",lw=.75,zorder=3)
    ax2.axvspan(28,40,facecolor="#F7CF74",edgecolor="#BD7A0D",
          hatch="///",alpha=.42,label="GPS outage (28–40 s)",zorder=1)
    for moment in (28,40):
        ax2.axvline(moment,color="#BD7A0D",lw=1.1,ls="--",zorder=5)
    ax2.set(xlabel="Time (s)",ylabel="East-position error (m)",
            xlim=(0,60))
    ax2.grid(alpha=.17)
    ax2.legend(loc="upper left",fontsize=12.8,framealpha=.93)
    ax2.set_title("(b) East-position error and uncertainty",
                  fontsize=19,fontweight="bold",pad=16)
    out = Path("../figures") if Path("../figures").is_dir() else Path(".")
    fig.savefig(out/"JNB8-KF.pdf",bbox_inches="tight",pad_inches=.17)
    fig.savefig(out/"JNB8-KF.png",dpi=220,bbox_inches="tight",pad_inches=.17)
    plt.show()